# Serial-v0 interface 살펴보기

원본 `gym-invmgmt`의 `reset` / `step`이 무엇을 돌려주는지 보는 노트북입니다.

- **원본 interface**: `gym.make`, `reset`, `step`이 반환하는 `obs`, `reward`, `terminated`, `truncated`, `info`
- **우리가 추가한 출력**: 그 값을 STATUS / ACTION / REWARD로 풀어 쓰는 부분

환경은 과제에 맞게 `GymInvMgmt/Serial-v0` + `use_goodwill=True`입니다.

사용 순서:
1. 설정 셀을 실행합니다.
2. `reset` 셀을 실행합니다.
3. `ACTION` 숫자를 바꾸고 `step` 셀을 실행합니다.
4. 같은 에피소드를 이어가려면 `step`만 다시 실행합니다.
5. 처음부터 보려면 `reset`을 다시 실행합니다.

In [1]:
from pathlib import Path
import sys

import gymnasium as gym
import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))

import gym_invmgmt  # noqa: F401
from walk_serial_steps import SERIAL_DEMAND_CONFIG, print_environment, print_period

print("gym_invmgmt", gym_invmgmt.__version__)
print("프로젝트 경로", ROOT)

gym_invmgmt 0.2.1
프로젝트 경로 /Users/oksjjj/coding/inventory-rl-project


## 1. 환경 만들기

원본 호출은 `gym.make`입니다. `demand_config`를 넘기면 기본값을 통째로 바꿉니다.

In [2]:
env = gym.make("GymInvMgmt/Serial-v0", demand_config=SERIAL_DEMAND_CONFIG)
core = env.unwrapped

print("환경 ID:", env.spec.id)
print("관측 공간:", env.observation_space)
print("행동 공간:", env.action_space)
print("행동 하한:", env.action_space.low)
print("행동 상한:", env.action_space.high)
print("행동 의미:")
print("  action[0]  유통 2 → 소매 1")
print("  action[1]  공장 3 → 유통 2")
print("  action[2]  원자재 4 → 공장 3")
print()
print_environment(core)

환경 ID: GymInvMgmt/Serial-v0
관측 공간: Box(-1000000.0, 1000000.0, (15,), float64)
행동 공간: Box(0.0, 3200.0, (3,), float64)
행동 하한: [0. 0. 0.]
행동 상한: [3200. 3200. 3200.]
행동 의미:
  action[0]  유통 2 → 소매 1
  action[1]  공장 3 → 유통 2
  action[2]  원자재 4 → 공장 3

=== 환경 GymInvMgmt/Serial-v0 ===
에피소드 길이: 30기간
관측 차원: 15
행동 차원: 3
원본 관측 STATUS 구성
  [직전 수요, 직전 미충족, 보유 재고 X, 이동 중 도착 일정, 기간 t, sentiment]
goodwill
  사용: True
  시작 sentiment: 1.0, 충족 시 ×1.01, 미충족 시 ×0.9
  범위: 0.2 ~ 2.0

노드
  0 (시장): 속성 없음
  1 (소매): {'I0': 100, 'h': 0.03}
  2 (유통): {'I0': 100, 'h': 0.02}
  3 (공장): {'I0': 200, 'C': 100, 'o': 0.01, 'v': 1.0, 'h': 0.01}
  4 (원자재): 속성 없음

링크
  1 -> 0: {'p': 2.0, 'b': 0.1, 'dist_param': {'mu': 20}}
  2 -> 1: {'L': 4, 'p': 1.5, 'g': 0.01}
  3 -> 2: {'L': 4, 'p': 1.0, 'g': 0.005}
  4 -> 3: {'L': 0, 'p': 0.5, 'g': 0.0}

행동 한 칸이 의미하는 발주 링크
  action[0] = 2(유통) -> 1(소매) 발주량, 리드타임 L=4
  action[1] = 3(공장) -> 2(유통) 발주량, 리드타임 L=4
  action[2] = 4(원자재) -> 3(공장) 발주량, 리드타임 L=0

원본 step() 순서
  1. 발주량을 공급 가능량으로 자른 뒤 R

## 2. reset

원본은 `(obs, info)`를 돌려줍니다. 에피소드를 처음부터 다시 보려면 이 셀만 다시 실행하면 됩니다.

In [4]:
SEED = 42
obs, info = env.reset(seed=SEED)

print("원본 reset 반환")
print("  obs 타입:", type(obs), "shape:", np.shape(obs))
print("  obs =", np.asarray(obs, dtype=float))
print("  info =", info)
print("  현재 기간 core.period =", core.period)

원본 reset 반환
  obs 타입: <class 'numpy.ndarray'> shape: (15,)
  obs = [  0.   0. 100. 100. 200.   0.   0.   0.   0.   0.   0.   0.   0.   0.
   1.]
  info = {}
  현재 기간 core.period = 0


In [5]:
def decode_obs(obs):
    """원본 관측 벡터를 칸별로 풀어 본다. 학습에는 이 함수가 필요 없다."""
    obs = np.asarray(obs, dtype=float).reshape(-1)
    return {
        "직전 수요": obs[0],
        "직전 미충족": obs[1],
        "재고 소매": obs[2],
        "재고 유통": obs[3],
        "재고 공장": obs[4],
        "파이프라인 2→1 (0~3기간 후)": obs[5:9],
        "파이프라인 3→2 (0~3기간 후)": obs[9:13],
        "기간 t": obs[13],
        "sentiment": obs[14],
    }

print("우리가 풀어 본 현재 STATUS")
for key, value in decode_obs(obs).items():
    print(f"  {key}: {value}")

우리가 풀어 본 현재 STATUS
  직전 수요: 0.0
  직전 미충족: 0.0
  재고 소매: 100.0
  재고 유통: 100.0
  재고 공장: 200.0
  파이프라인 2→1 (0~3기간 후): [0. 0. 0. 0.]
  파이프라인 3→2 (0~3기간 후): [0. 0. 0. 0.]
  기간 t: 0.0
  sentiment: 1.0


## 3. action 넣고 step

아래 숫자만 바꾼 뒤 `step` 셀을 실행합니다.

- `action[0]`: 유통 → 소매
- `action[1]`: 공장 → 유통
- `action[2]`: 원자재 → 공장

In [6]:
ACTION = [20.0, 20.0, 20.0]
action = np.asarray(ACTION, dtype=np.float64)
print("이번에 넣을 action =", action)

이번에 넣을 action = [20. 20. 20.]


In [11]:
period_before = core.period
obs, reward, terminated, truncated, info = env.step(action)

print("원본 step 반환")
print("  obs shape:", np.shape(obs))
print("  obs =", np.asarray(obs, dtype=float))
print("  reward =", float(reward))
print("  terminated =", terminated)
print("  truncated =", truncated)
print("  info =", info)
print()
print("우리가 풀어 본 다음 STATUS")
for key, value in decode_obs(obs).items():
    print(f"  {key}: {value}")
print()
print_period(core, period_before, action, float(reward), info)

원본 step 반환
  obs shape: (15,)
  obs = [ 19.           0.          13.          20.         200.
  20.          20.          20.          20.          20.
  20.          20.          20.           5.           1.04060401]
  reward = 23.81
  terminated = False
  truncated = False
  info = {'period': 4, 'step_reward': 23.81, 'total_inventory': np.float64(233.0), 'total_backlog': np.float64(0.0), 'sentiment': 1.04060401}

우리가 풀어 본 다음 STATUS
  직전 수요: 19.0
  직전 미충족: 0.0
  재고 소매: 13.0
  재고 유통: 20.0
  재고 공장: 200.0
  파이프라인 2→1 (0~3기간 후): [20. 20. 20. 20.]
  파이프라인 3→2 (0~3기간 후): [20. 20. 20. 20.]
  기간 t: 5.0
  sentiment: 1.04060401


기간 4

[STATUS]
------------------------------------------------------------
원본 관측이 담는 값
  직전 수요: 14.00
  직전 미충족: 0.00
  기간 t: 4
  sentiment: 1.030
보유 재고 X[t]
  노드 1 (소매): 12.00
  노드 2 (유통): 20.00
  노드 3 (공장): 200.00
이동 중 재고 Y[t]
  2->1 (L=4): 80.00
  3->2 (L=4): 80.00
  4->3 (L=0): 0.00
도착 일정 (관측 파이프라인, 0기간 후 = 이번 기간 입고)
  2->1: 합 80.00  [0기간 후 20.00, 1기간 후 20.00, 2

## 사용 방법

- 다른 발주량을 보려면 `ACTION = [10, 30, 20]`처럼 바꾸고 `ACTION` 셀과 `step` 셀을 다시 실행합니다.
- 30기간이 끝나면 `truncated=True`가 됩니다. 그때는 `reset`부터 다시 실행합니다.
- 학습 코드가 쓰는 값은 `obs`, `reward`, `terminated`, `truncated`입니다. 아래 STATUS/REWARD 산식은 이해를 위한 출력입니다.